<a href="https://colab.research.google.com/github/Math-BUG/INF-494/blob/main/00_datasets_e_configuracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 - Datasets e configuracao experimental comum

Este notebook cria o protocolo comum usado pelos notebooks de estrategia. Ele gera datasets sinteticos e amostras reais, normaliza tudo para `float32`, estima `eps` pelo k-esimo vizinho e salva `data/manifest.csv` e `results/resumo_datasets.csv`.

Os arquivos `.bin` e `.npy` sao gerados no Colab e nao precisam ser commitados. O repositorio mantem apenas o notebook e um `manifest_exemplo.csv`.

In [13]:
# Diagnostico do ambiente do Colab.
# Ative GPU em: Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU.
!nvidia-smi
!nvcc --version
!which nvprof || echo "nvprof nao encontrado"

Mon Jun  8 06:03:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import make_blobs, make_moons, fetch_covtype, fetch_kddcup99, fetch_openml
from sklearn.neighbors import NearestNeighbors

SEED = 42
np.random.seed(SEED)

# Use Google Drive para persistir data/ e results/ entre notebooks e runtimes do Colab.
# Se nao estiver no Colab, o codigo cai automaticamente para o diretorio local.
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/INF494_DBSCAN")

BASE_DIR = Path(".")
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = DRIVE_PROJECT_DIR
    except Exception as exc:
        print("Google Drive nao montado; usando diretorio local.")
        print("Motivo:", repr(exc))
        BASE_DIR = Path(".")

DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/INF494_DBSCAN
DATA_DIR: /content/drive/MyDrive/INF494_DBSCAN/data
RESULTS_DIR: /content/drive/MyDrive/INF494_DBSCAN/results


In [15]:
# Modos padronizados.
RUN_MODE = "benchmark"  # "quick" ou "benchmark"

N_SAMPLES_QUICK = 4000
N_SAMPLES_LIST_BENCHMARK = [10000, 25000, 50000, 100000]

N_SAMPLES_LIST = [N_SAMPLES_QUICK] if RUN_MODE == "quick" else N_SAMPLES_LIST_BENCHMARK
MIN_SAMPLES = 8
EPS_QUANTILE = 0.90
EPS_SAMPLE_SIZE = 5000
INCLUIR_DATASETS_REAIS = True
INCLUIR_HIGGS = True  # opcional; pode falhar por tamanho/rede no Colab.

print("RUN_MODE:", RUN_MODE)
print("N_SAMPLES_LIST:", N_SAMPLES_LIST)

RUN_MODE: benchmark
N_SAMPLES_LIST: [10000, 25000, 50000, 100000]


In [16]:
def normalize_minmax(X):
    X = np.asarray(X, dtype=np.float32)
    mn = X.min(axis=0)
    mx = X.max(axis=0)
    denom = mx - mn
    denom[denom == 0.0] = 1.0
    return ((X - mn) / denom).astype(np.float32)


def estimate_eps(X, min_samples=8, quantile=0.90, sample_size=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=np.float32)
    if len(X) > sample_size:
        idx = rng.choice(len(X), size=sample_size, replace=False)
        Xs = X[idx]
    else:
        Xs = X
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(Xs)
    dists, _ = nn.kneighbors(Xs)
    return float(np.quantile(dists[:, -1], quantile))


def sample_rows(X, y, n_samples, seed=SEED):
    X = np.asarray(X)
    y = np.asarray(y)
    n = min(int(n_samples), len(X))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=n, replace=False) if len(X) > n else np.arange(len(X))
    return X[idx], y[idx]


def factorize_mixed_matrix(X):
    df = pd.DataFrame(X)
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda v: v.decode("utf-8", errors="ignore") if isinstance(v, bytes) else v)
            df[col] = pd.factorize(df[col])[0].astype(np.float32)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype(np.float32)
    return df.to_numpy(dtype=np.float32)


def factorize_labels(y):
    y = pd.Series(y)
    y = y.apply(lambda v: v.decode("utf-8", errors="ignore") if isinstance(v, bytes) else v)
    return pd.factorize(y)[0].astype(np.int32)

In [17]:
def make_synthetic_dataset(name, n_samples, seed=SEED):
    rng = np.random.default_rng(seed)
    if name == "dense_blobs_2d":
        X, y = make_blobs(n_samples=n_samples, centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)], cluster_std=0.18, random_state=seed)
        desc = "Blobs 2D densos"
        obj = "Testar gargalo com muitos core points."
    elif name == "heterogeneous_blobs_2d":
        n1 = int(0.45 * n_samples)
        n2 = int(0.35 * n_samples)
        n3 = n_samples - n1 - n2
        X1, _ = make_blobs(n_samples=n1, centers=[(-4, 0)], cluster_std=0.08, random_state=seed + 1)
        X2, _ = make_blobs(n_samples=n2, centers=[(0, 0)], cluster_std=0.25, random_state=seed + 2)
        X3, _ = make_blobs(n_samples=n3, centers=[(4, 0)], cluster_std=0.65, random_state=seed + 3)
        X = np.vstack([X1, X2, X3])
        y = np.concatenate([np.zeros(n1, dtype=np.int32), np.ones(n2, dtype=np.int32), np.full(n3, 2, dtype=np.int32)])
        desc = "Blobs 2D com densidades diferentes"
        obj = "Testar regioes com densidades diferentes."
    elif name == "dense_blobs_noise_2d":
        n_noise = int(0.20 * n_samples)
        n_blob = n_samples - n_noise
        X_blob, y_blob = make_blobs(n_samples=n_blob, centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)], cluster_std=0.20, random_state=seed)
        noise = rng.uniform(low=-5.5, high=5.5, size=(n_noise, 2))
        X = np.vstack([X_blob, noise])
        y = np.concatenate([y_blob.astype(np.int32), np.full(n_noise, -1, dtype=np.int32)])
        desc = "Blobs 2D densos com ruido/outliers"
        obj = "Testar ruido e outliers."
    elif name == "moons_2d":
        X, y = make_moons(n_samples=n_samples, noise=0.045, random_state=seed)
        desc = "Duas luas 2D"
        obj = "Testar formato nao convexo."
    elif name == "blobs_32d":
        X, y = make_blobs(n_samples=n_samples, centers=6, n_features=32, cluster_std=0.45, random_state=seed)
        desc = "Blobs sinteticos com 32 dimensoes"
        obj = "Testar quantizacao e leitura de memoria."
    else:
        raise ValueError(name)
    return X, y, desc, obj


def load_real_source(name):
    if name == "real_covtype_sample":
        data = fetch_covtype(download_if_missing=True)
        X = data.data
        y = data.target.astype(np.int32)
        return X, y, "Amostra do Covertype", "Dataset real maior para testar amostragem controlada."
    if name == "real_kddcup99_sample":
        data = fetch_kddcup99(percent10=True, shuffle=True, random_state=SEED, download_if_missing=True)
        X = factorize_mixed_matrix(data.data)
        y = factorize_labels(data.target)
        return X, y, "Amostra do KDDCup99", "Dataset real com atributos mistos convertidos para numerico."
    if name == "real_higgs_sample":
        data = fetch_openml("higgs", version=1, as_frame=True, parser="auto")
        frame = data.frame
        y = factorize_labels(frame.iloc[:, 0].to_numpy())
        X = frame.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        return X, y, "Amostra opcional do HIGGS", "Dataset real maior opcional; pode falhar por rede/tamanho."
    raise ValueError(name)

In [18]:
synthetic_names = ["dense_blobs_2d", "heterogeneous_blobs_2d", "dense_blobs_noise_2d", "moons_2d", "blobs_32d"]
real_names = ["real_covtype_sample", "real_kddcup99_sample"]
if INCLUIR_HIGGS:
    real_names.append("real_higgs_sample")

rows = []

def save_dataset_entry(dataset_name, X, y, n_target, tipo, descricao, objetivo, observacao):
    Xs, ys = sample_rows(X, y, n_target, seed=SEED + int(n_target))
    Xs = normalize_minmax(Xs)
    ys = np.asarray(ys, dtype=np.int32)
    n, d = Xs.shape
    eps = estimate_eps(Xs, min_samples=MIN_SAMPLES, quantile=EPS_QUANTILE, sample_size=EPS_SAMPLE_SIZE)
    data_path = DATA_DIR / f"{dataset_name}_{n}_f32.bin"
    label_path = DATA_DIR / f"{dataset_name}_{n}_y.npy"
    Xs.tofile(data_path)
    np.save(label_path, ys)
    rows.append({
        "dataset_name": dataset_name,
        "n_samples": n,
        "n_features": d,
        "data_path": str(data_path),
        "label_path": str(label_path),
        "eps": eps,
        "min_samples": MIN_SAMPLES,
        "tipo": tipo,
        "descricao": descricao,
        "objetivo": objetivo,
        "observacao": observacao,
    })
    print(f"Salvo: {dataset_name} n={n} d={d} eps={eps:.6f}")


for dataset_name in synthetic_names:
    for n_samples in N_SAMPLES_LIST:
        X, y, descricao, objetivo = make_synthetic_dataset(dataset_name, n_samples)
        save_dataset_entry(dataset_name, X, y, n_samples, "sintetico", descricao, objetivo, "Gerado no notebook 00.")

if INCLUIR_DATASETS_REAIS:
    for dataset_name in real_names:
        try:
            X_full, y_full, descricao, objetivo = load_real_source(dataset_name)
            for n_samples in N_SAMPLES_LIST:
                save_dataset_entry(dataset_name, X_full, y_full, n_samples, "real_amostrado", descricao, objetivo, "Amostra controlada; dados brutos nao sao commitados.")
        except Exception as exc:
            print(f"Nao foi possivel carregar {dataset_name}: {repr(exc)}")

manifest = pd.DataFrame(rows)
manifest_path = DATA_DIR / "manifest.csv"
summary_path = RESULTS_DIR / "resumo_datasets.csv"
manifest.to_csv(manifest_path, index=False)
manifest.to_csv(summary_path, index=False)
print("Manifest salvo em:", manifest_path)
print("Resumo salvo em:", summary_path)
display(manifest)

Salvo: dense_blobs_2d n=10000 d=2 eps=0.008030
Salvo: dense_blobs_2d n=25000 d=2 eps=0.007624
Salvo: dense_blobs_2d n=50000 d=2 eps=0.007737
Salvo: dense_blobs_2d n=100000 d=2 eps=0.007831
Salvo: heterogeneous_blobs_2d n=10000 d=2 eps=0.017316
Salvo: heterogeneous_blobs_2d n=25000 d=2 eps=0.016772
Salvo: heterogeneous_blobs_2d n=50000 d=2 eps=0.016531
Salvo: heterogeneous_blobs_2d n=100000 d=2 eps=0.016057
Salvo: dense_blobs_noise_2d n=10000 d=2 eps=0.045833
Salvo: dense_blobs_noise_2d n=25000 d=2 eps=0.045502
Salvo: dense_blobs_noise_2d n=50000 d=2 eps=0.045627
Salvo: dense_blobs_noise_2d n=100000 d=2 eps=0.044835
Salvo: moons_2d n=10000 d=2 eps=0.014299
Salvo: moons_2d n=25000 d=2 eps=0.013842
Salvo: moons_2d n=50000 d=2 eps=0.013895
Salvo: moons_2d n=100000 d=2 eps=0.014165
Salvo: blobs_32d n=10000 d=32 eps=0.180816
Salvo: blobs_32d n=25000 d=32 eps=0.178040
Salvo: blobs_32d n=50000 d=32 eps=0.175853
Salvo: blobs_32d n=100000 d=32 eps=0.175033
Salvo: real_covtype_sample n=10000 d=54

,dataset_name,n_samples,n_features,data_path,label_path,eps,min_samples,tipo,descricao,objetivo,observacao
0,dense_blobs_2d,10000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.008030,8,sintetico,Blobs 2D densos,Testar gargalo com muitos core points.,Gerado no notebook 00.
1,dense_blobs_2d,25000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.007624,8,sintetico,Blobs 2D densos,Testar gargalo com muitos core points.,Gerado no notebook 00.
2,dense_blobs_2d,50000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.007737,8,sintetico,Blobs 2D densos,Testar gargalo com muitos core points.,Gerado no notebook 00.
3,dense_blobs_2d,100000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.007831,8,sintetico,Blobs 2D densos,Testar gargalo com muitos core points.,Gerado no notebook 00.
4,heterogeneous_blobs_2d,10000,2,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,0.017316,8,sintetico,Blobs 2D com densidades diferentes,Testar regioes com densidades diferentes.,Gerado no notebook 00.
5,heterogeneous_blobs_2d,25000,2,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,0.016772,8,sintetico,Blobs 2D com densidades diferentes,Testar regioes com densidades diferentes.,Gerado no notebook 00.
6,heterogeneous_blobs_2d,50000,2,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,0.016531,8,sintetico,Blobs 2D com densidades diferentes,Testar regioes com densidades diferentes.,Gerado no notebook 00.
7,heterogeneous_blobs_2d,100000,2,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,/content/drive/MyDrive/INF494_DBSCAN/data/hete...,0.016057,8,sintetico,Blobs 2D com densidades diferentes,Testar regioes com densidades diferentes.,Gerado no notebook 00.
8,dense_blobs_noise_2d,10000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.045833,8,sintetico,Blobs 2D densos com ruido/outliers,Testar ruido e outliers.,Gerado no notebook 00.
9,dense_blobs_noise_2d,25000,2,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,/content/drive/MyDrive/INF494_DBSCAN/data/dens...,0.045502,8,sintetico,Blobs 2D densos com ruido/outliers,Testar ruido e outliers.,Gerado no notebook 00.


## Observacao

Os notebooks 01, 02, 03 e 04 usam `data/manifest.csv`. Se esse arquivo nao existir, eles podem gerar um dataset sintetico pequeno apenas para depuracao, mas os resultados finais devem vir deste protocolo comum.